# 09c - Individual base-learner tuning (XGBoost + RF)

Second of three tuning stages: feature set (`09b`), individual base-learner hyperparameters here, ensemble weights last (`09d`) - tuning happens before ensembling, so the ensemble-weight decision in `09d` only has to be made once, on the best available version of each learner.

Random search for XGBoost and RF, scored on the leak-free `train_sub` (year < 2024) / `val` (year == 2024) split, on the finalized feature set - `GBM_FEATURES` plus the stacked lateness feature from `09b` (`pred_lateness_gbm`), rebuilt here scoped to `train_sub`/`val`. Stops at picking the best hyperparameters - doesn't touch ensemble weights, that's `09d`'s job, done once on these tuned models.

## Setup

In [ ]:
import time as _time

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    HistGradientBoostingRegressor, RandomForestClassifier,
)
from sklearn.model_selection import ParameterSampler, KFold
from sklearn.metrics import roc_auc_score

import xgboost as xgb

from utils import (
    load_model_split, prep_gbm_matrices, GBM_FEATURES, GBM_CAT_FEATURES,
)

BASEPATH = "../data"

# temporary progress instrumentation for long-running cells below # Claude
PROGRESS_LOG = "/tmp/09c_progress.log"

def log_progress(msg):
    ts = _time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(PROGRESS_LOG, "a") as f:
        f.write(line + "\n")

log_progress("09c_tuning.ipynb started")

train_df, test_df = load_model_split(BASEPATH)

# same temporal sub split that ensembling in 09d 
# and error analysis in 09e will use.
# never let tuning see the actual test set
VAL_YEAR = 2024
train_sub = train_df[train_df["year"] < VAL_YEAR].copy()
val_df = train_df[train_df["year"] == VAL_YEAR].copy()

y_train_sub_otp = train_sub["is_otp"]
y_val_otp = val_df["is_otp"]
y_train_sub_log = np.log1p(train_sub["lateness"])

print(
    f"train_sub: {train_sub.shape}, val: {val_df.shape}, "
    f"test: {test_df.shape}"
)

[09:03:44] 09c_tuning.ipynb started


train_sub: (1170530, 107), val: (187033, 107), test: (126332, 107)


## 0. Rebuild the stacked lateness feature (train_sub/val scope)

Same idea as `09b` but scoped to `train_sub`/`val` instead of the full training set, since the search below is scored on `val` and should never see anything derived from `val`/`test`. 

Uses `09b`'s tuned regression-GBM hyperparameters as a fixed configuration vs. re-tuning here.
This performs well so `09d` and `09e` rebuild it again, scoped to `train_sub`/`val`/`test`, for the final model.

In [2]:
# tuned regression-GBM hyperparameters from 09b (RandomizedSearchCV on the
# full training set) -- reused here as a fixed config, not re-searched
TUNED_REG_PARAMS = dict(
    min_samples_leaf = 10, max_leaf_nodes = 127, max_iter = 500,
    max_depth = 7, learning_rate = 0.03, l2_regularization = 1,
)

X_train_sub_gbm, X_val_gbm = prep_gbm_matrices(train_sub, val_df)

_t0 = _time.time()
log_progress(
    "Rebuilding out-of-fold stacked lateness feature on "
    "train_sub (5-fold KFold)..."
)
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
oof_pred_lateness = np.zeros(len(train_sub))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_sub_gbm), start = 1):
    _tfold = _time.time()
    fold_reg = HistGradientBoostingRegressor(
        categorical_features = "from_dtype", random_state = 42,
        **TUNED_REG_PARAMS
    )
    fold_reg.fit(X_train_sub_gbm.iloc[tr_idx], y_train_sub_log.iloc[tr_idx])
    oof_pred_lateness[val_idx] = np.expm1(
        fold_reg.predict(X_train_sub_gbm.iloc[val_idx])
    )
    log_progress(f"  fold {fold}/5 done ({_time.time() - _tfold:.1f}s)")

train_sub["pred_lateness_gbm"] = oof_pred_lateness

reg_full = HistGradientBoostingRegressor(
    categorical_features = "from_dtype", random_state = 42,
    **TUNED_REG_PARAMS
)
reg_full.fit(X_train_sub_gbm, y_train_sub_log)
val_df["pred_lateness_gbm"] = np.expm1(reg_full.predict(X_val_gbm))
log_progress(
    f"Stacked feature rebuilt on train_sub/val "
    f"({_time.time() - _t0:.1f}s total)"
)

STACKED_FEATURES = GBM_FEATURES + ["pred_lateness_gbm"]

[09:03:46] Rebuilding out-of-fold stacked lateness feature on train_sub (5-fold KFold)...


[09:04:34]   fold 1/5 done (47.4s)


[09:05:22]   fold 2/5 done (48.4s)


[09:06:13]   fold 3/5 done (50.8s)


[09:07:02]   fold 4/5 done (49.4s)


[09:07:44]   fold 5/5 done (41.7s)


[09:08:31] Stacked feature rebuilt on train_sub/val (284.5s total)


## 1. XGBoost random search

15 randomly sampled configurations plus the exact default config used everywhere else, all fit on `train_sub` (now including the stacked lateness feature) and scored by ROC-AUC on the full `val` set.

In [3]:
xgb_default_params = dict(
    n_estimators = 100, max_depth = 6, learning_rate = 0.3,
    subsample = 1.0, colsample_bytree = 1.0, min_child_weight = 1,
    reg_lambda = 1,
)
xgb_param_grid = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.03, 0.05, 0.1, 0.15, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "reg_lambda": [0.5, 1, 2, 5],
}
xgb_candidates = [("default", xgb_default_params)] + [
    ("search", p)
    for p in ParameterSampler(xgb_param_grid, n_iter = 15, random_state = 42)
]

X_train_sub_stacked, X_val_stacked = prep_gbm_matrices(
    train_sub, val_df,
    features = STACKED_FEATURES, cat_features = GBM_CAT_FEATURES,
)

_t0 = _time.time()
log_progress(
    f"Running XGBoost random search ({len(xgb_candidates)} candidates)..."
)
xgb_search_rows = []
best_xgb_model, best_xgb_params = None, None
best_xgb_auc, best_xgb_tag = -1, None
default_xgb_model, default_xgb_auc = None, None
for i, (tag, params) in enumerate(xgb_candidates, start = 1):
    _tc = _time.time()
    model = xgb.XGBClassifier(
        random_state = 42, enable_categorical = True,
        tree_method = "hist", **params
    )
    model.fit(X_train_sub_stacked, y_train_sub_otp)
    p_val = model.predict_proba(X_val_stacked)[:, 1]
    auc = roc_auc_score(y_val_otp, p_val)
    xgb_search_rows.append({"tag": tag, **params, "val_roc_auc": auc})
    log_progress(
        f"  [{i}/{len(xgb_candidates)}] {tag}: "
        f"val_roc_auc={auc:.4f} ({_time.time() - _tc:.1f}s)"
    )
    if tag == "default":
        default_xgb_model, default_xgb_auc = model, auc
    if auc > best_xgb_auc:
        best_xgb_model = model
        best_xgb_params, best_xgb_auc, best_xgb_tag = params, auc, tag
log_progress(f"XGBoost search complete ({_time.time() - _t0:.1f}s total)")

xgb_search_df = pd.DataFrame(xgb_search_rows).sort_values(
    "val_roc_auc", ascending = False
)
print(xgb_search_df.to_string(index = False))
print(f"\ndefault XGBoost val ROC-AUC: {default_xgb_auc:.4f}")
print(
    f"best XGBoost val ROC-AUC:    {best_xgb_auc:.4f} "
    f"({best_xgb_tag}, params: {best_xgb_params})"
)

[09:08:31] Running XGBoost random search (16 candidates)...


[09:08:37]   [1/16] default: val_roc_auc=0.8161 (5.3s)


[09:08:45]   [2/16] search: val_roc_auc=0.8170 (8.0s)


[09:09:07]   [3/16] search: val_roc_auc=0.8179 (22.2s)


[09:09:18]   [4/16] search: val_roc_auc=0.8166 (10.9s)


...

[09:10:39]   [12/16] search: val_roc_auc=0.8159 (8.7s)


[09:10:45]   [13/16] search: val_roc_auc=0.8165 (5.5s)


[09:10:52]   [14/16] search: val_roc_auc=0.8159 (7.6s)


[09:10:58]   [15/16] search: val_roc_auc=0.8159 (6.1s)


[09:11:15]   [16/16] search: val_roc_auc=0.8180 (17.1s)
[09:11:15] XGBoost search complete (164.1s total)
    tag  n_estimators  max_depth  learning_rate  subsample  colsample_bytree  min_child_weight  reg_lambda  val_roc_auc
 search           300          7           0.05        1.0               1.0                 1         5.0     0.817989
 search           400          7           0.03        1.0               0.6                 3         2.0     0.817890
 search           200          7           0.03        0.8               0.8                10         5.0     0.817044
 search           200          5           0.10        0.8               0.8                10         5.0     0.817001
 search           300          5           0.05        0.8               0.8                 5         0.5     0.816982
 search           200          5           0.10        0.6               1.0                 3         0.5     0.816819
 search           300          7           0.15       

## 2. Random forest random search

8 randomly sampled configurations plus the current default

Less candidates than XGBoost search bc each RF fit is way slower/more expensive and baselines lean toward XGB anyway, I don't want to spend too long on this

In [4]:
X_train_sub_rf = pd.get_dummies(
    train_sub[STACKED_FEATURES], columns = GBM_CAT_FEATURES
)
X_val_rf = pd.get_dummies(
    val_df[STACKED_FEATURES], columns = GBM_CAT_FEATURES
).reindex(columns = X_train_sub_rf.columns, fill_value = 0)

rf_default_params = dict(
    n_estimators = 200, max_depth = 20, min_samples_leaf = 20,
    max_features = "sqrt",
)
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 15, 20, 25, None],
    "min_samples_leaf": [5, 10, 20, 50],
    "max_features": ["sqrt", 0.5, None],
}
rf_candidates = [("default", rf_default_params)] + [
    ("search", p)
    for p in ParameterSampler(rf_param_grid, n_iter = 8, random_state = 42)
]

_t0 = _time.time()
log_progress(
    f"Running RF random search ({len(rf_candidates)} candidates) "
    f"-- slowest cell in this notebook..."
)
rf_search_rows = []
best_rf_model, best_rf_params = None, None
best_rf_auc, best_rf_tag = -1, None
default_rf_model, default_rf_auc = None, None
for i, (tag, params) in enumerate(rf_candidates, start = 1):
    _tc = _time.time()
    model = RandomForestClassifier(n_jobs = -1, random_state = 42, **params)
    model.fit(X_train_sub_rf, y_train_sub_otp)
    p_val = model.predict_proba(X_val_rf)[:, 1]
    auc = roc_auc_score(y_val_otp, p_val)
    rf_search_rows.append({"tag": tag, **params, "val_roc_auc": auc})
    log_progress(
        f"  [{i}/{len(rf_candidates)}] {tag}: "
        f"val_roc_auc={auc:.4f} ({_time.time() - _tc:.1f}s)"
    )
    if tag == "default":
        default_rf_model, default_rf_auc = model, auc
    if auc > best_rf_auc:
        best_rf_model = model
        best_rf_params, best_rf_auc, best_rf_tag = params, auc, tag
log_progress(f"RF search complete ({_time.time() - _t0:.1f}s total)")

rf_search_df = pd.DataFrame(rf_search_rows).sort_values(
    "val_roc_auc", ascending = False
)
print(rf_search_df.to_string(index = False))
print(f"\ndefault RF val ROC-AUC: {default_rf_auc:.4f}")
print(
    f"best RF val ROC-AUC:    {best_rf_auc:.4f} "
    f"({best_rf_tag}, params: {best_rf_params})"
)

[09:11:16] Running RF random search (9 candidates) -- slowest cell in this notebook...


[09:12:44]   [1/9] default: val_roc_auc=0.8125 (88.7s)


[09:17:24]   [2/9] search: val_roc_auc=0.8154 (280.0s)


[09:18:00]   [3/9] search: val_roc_auc=0.8109 (35.1s)


[09:18:45]   [4/9] search: val_roc_auc=0.8124 (45.8s)


[09:19:30]   [5/9] search: val_roc_auc=0.8121 (45.0s)


[09:21:19]   [6/9] search: val_roc_auc=0.8115 (108.9s)


[09:24:11]   [7/9] search: val_roc_auc=0.8152 (171.4s)


[09:29:43]   [8/9] search: val_roc_auc=0.8146 (331.9s)


[09:53:45]   [9/9] search: val_roc_auc=0.8154 (1442.8s)
[09:53:45] RF search complete (2549.7s total)
    tag  n_estimators  max_depth  min_samples_leaf max_features  val_roc_auc
 search           200       10.0                20          0.5     0.815432
 search           300       15.0                20         None     0.815369
 search           100       10.0                10          0.5     0.815235
 search           100       10.0                 5         None     0.814645
default           200       20.0                20         sqrt     0.812537
 search           100        NaN                50         sqrt     0.812388
 search           100       20.0                20         sqrt     0.812150
 search           200        NaN                 5         sqrt     0.811472
 search           100       15.0                20         sqrt     0.810909

default RF val ROC-AUC: 0.8125
best RF val ROC-AUC:    0.8154 (search, params: {'n_estimators': 200, 'min_samples_leaf': 20, 'm

## Summary

In [5]:
print("=== Tuning summary ===")
print(
    f"XGBoost: default val AUC={default_xgb_auc:.4f}, "
    f"best val AUC={best_xgb_auc:.4f} ({best_xgb_tag})"
)
print(f"  TUNED_XGB_PARAMS = {best_xgb_params}")
print(
    f"RF:      default val AUC={default_rf_auc:.4f}, "
    f"best val AUC={best_rf_auc:.4f} ({best_rf_tag})"
)
print(f"  TUNED_RF_PARAMS = {best_rf_params}")
log_progress("09c_tuning.ipynb complete")

=== Tuning summary ===
XGBoost: default val AUC=0.8161, best val AUC=0.8180 (search)
  TUNED_XGB_PARAMS = {'subsample': 1.0, 'reg_lambda': 5, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
RF:      default val AUC=0.8125, best val AUC=0.8154 (search)
  TUNED_RF_PARAMS = {'n_estimators': 200, 'min_samples_leaf': 20, 'max_features': 0.5, 'max_depth': 10}
[09:53:45] 09c_tuning.ipynb complete
